## Library Management Database

In [ ]:
# Install Faker to generate realistic synthetic data
!pip install Faker

# Import required libraries
import sqlite3
import random
import pandas as pd

from faker import Faker
from datetime import date, timedelta
from google.colab import files

# Initialise Faker
fake = Faker()

# Set random seed to generate reproducible data
random.seed(42)
Faker.seed(42)

print("Libraries imported successfully")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.1/2.1 MB 21.0 MB/s  0:00:00
Libraries imported successfully


Database Creation using SQLite

In [ ]:
# Create a new SQLite database
db_file = "Library_Management_Final.db"

conn = sqlite3.connect(db_file)

# Enable foreign key constraints
conn.execute("PRAGMA foreign_keys = ON")

# Create SQL cursor
cursor = conn.cursor()

print("Library Management Database created successfully")

Library Management Database created successfully


Books Table creation

In [ ]:
# Create the Books table
cursor.execute("""
    CREATE TABLE Books (
        book_id INTEGER PRIMARY KEY AUTOINCREMENT,
        title TEXT NOT NULL,
        author_name TEXT NOT NULL,
        isbn TEXT UNIQUE NOT NULL,
        genre TEXT NOT NULL,
        publication_year INTEGER,
        pages INTEGER NOT NULL,
        rating REAL,

        CHECK (publication_year BETWEEN 1450 AND 2026),
        CHECK (pages > 0),
        CHECK (rating BETWEEN 0 AND 5)
    )
""")

conn.commit()

print("Books table created successfully")

Books table created successfully


Members Table creation

In [ ]:
# Create the Members table
cursor.execute("""
    CREATE TABLE Members (
        member_id INTEGER PRIMARY KEY AUTOINCREMENT,
        member_name TEXT NOT NULL,
        email TEXT UNIQUE NOT NULL,
        phone TEXT,
        membership_type TEXT NOT NULL,
        join_date TEXT NOT NULL,
        age INTEGER NOT NULL,
        city TEXT NOT NULL,

        CHECK (membership_type IN ('Basic', 'Silver', 'Gold')),
        CHECK (age BETWEEN 18 AND 100)
    )
""")

conn.commit()

print("Members table created successfully")

Members table created successfully


Borrowing Table creation

In [ ]:
# Create the Borrowings table
cursor.execute("""
    CREATE TABLE Borrowings (
        book_id INTEGER NOT NULL,
        member_id INTEGER NOT NULL,
        borrow_date TEXT NOT NULL,
        due_date TEXT NOT NULL,
        return_date TEXT,
        fine_amount REAL DEFAULT 0,
        borrowing_status TEXT NOT NULL,
        renewal_count INTEGER DEFAULT 0,

        PRIMARY KEY (book_id, member_id, borrow_date),

        FOREIGN KEY (book_id)
            REFERENCES Books(book_id),

        FOREIGN KEY (member_id)
            REFERENCES Members(member_id),

        CHECK (fine_amount >= 0),
        CHECK (borrowing_status IN ('Borrowed', 'Returned', 'Overdue')),
        CHECK (renewal_count >= 0)
    )
""")

conn.commit()

print("Borrowings table created successfully")

Borrowings table created successfully


Generating The Data

In [ ]:
# Generate realistic book information

genres = [
    "Fiction", "Science", "History", "Technology",
    "Biography", "Mystery", "Fantasy", "Romance",
    "Education", "Philosophy"
]

books_data = []

# Generate 300 books
for i in range(300):

    book = (
        fake.sentence(nb_words=4).replace(".", ""),
        fake.name(),
        fake.unique.isbn13(),
        random.choice(genres),
        random.randint(1990, 2026),
        random.randint(100, 800),
        round(random.uniform(2.5, 5.0), 1)
    )

    books_data.append(book)


# Insert books into the database
cursor.executemany("""
    INSERT INTO Books (
        title,
        author_name,
        isbn,
        genre,
        publication_year,
        pages,
        rating
    )
    VALUES (?, ?, ?, ?, ?, ?, ?)
""", books_data)

conn.commit()

# Display generated book records
books_df = pd.read_sql_query("SELECT * FROM Books", conn)

print("Books generated successfully!")
print("Total Books:", len(books_df))

display(books_df.head(10))

Books generated successfully!
Total Books: 300


,book_id,title,author_name,isbn,genre,publication_year,pages,rating
0,1,Agent every,Noah Rhodes,978-1-00-133890-3,Science,1991,381,3.1
1,2,Current practice nation determine operation,Abigail Shaffer,978-1-4235-1161-8,History,1996,792,4.4
2,3,Suggest page southern role,Valerie Gray,978-1-931034-13-5,Education,1995,704,3.6
3,4,Herself law,Carla Kelly,978-0-928327-64-9,Fiction,1995,323,3.1
4,5,Detail food shoulder argue start,Jennifer Jones,978-0-7672-4238-7,Philosophy,1991,674,3.0
5,6,Enjoy once today human despite,Ms. Ann Williams MD,978-1-916697-84-3,Education,2016,325,3.6
6,7,Team activity result race Mr,Martin Rodriguez,978-0-7048-2814-8,Biography,1990,263,4.2
7,8,Participant check several much,Kerry Smith,978-1-01-543039-6,Mystery,2007,259,3.0
8,9,Beat magazine,Philip Cannon,978-1-82489-638-3,Mystery,1996,194,3.4
9,10,Require human public,Steven Hayes,978-0-15-098393-4,Mystery,2012,718,3.2


In [ ]:
# Check book records for missing and duplicate values

cursor.execute("SELECT COUNT(*) FROM Books")
total_books = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(DISTINCT isbn) FROM Books")
unique_isbn = cursor.fetchone()[0]

cursor.execute("""
    SELECT COUNT(*)
    FROM Books
    WHERE title IS NULL
       OR author_name IS NULL
       OR isbn IS NULL
       OR genre IS NULL
       OR pages IS NULL
""")

missing_values = cursor.fetchone()[0]

print("Total Books:", total_books)
print("Unique ISBNs:", unique_isbn)
print("Records with missing mandatory values:", missing_values)

Total Books: 300
Unique ISBNs: 300
Records with missing mandatory values: 0


In [ ]:
# Generate realistic library member information

membership_types = ["Basic", "Silver", "Gold"]

members_data = []

# Generate 250 members
for i in range(250):

    member = (
        fake.name(),
        fake.unique.email(),
        fake.phone_number(),
        random.choice(membership_types),
        fake.date_between(start_date="-5y", end_date="today").isoformat(),
        random.randint(18, 75),
        fake.city()
    )

    members_data.append(member)


# Insert members into the database
cursor.executemany("""
    INSERT INTO Members (
        member_name,
        email,
        phone,
        membership_type,
        join_date,
        age,
        city
    )
    VALUES (?, ?, ?, ?, ?, ?, ?)
""", members_data)

conn.commit()

# Display member records
members_df = pd.read_sql_query("SELECT * FROM Members", conn)

print("Members generated successfully")
print("Total Members:", len(members_df))

display(members_df.head(10))

Members generated successfully!
Total Members: 250


,member_id,member_name,email,phone,membership_type,join_date,age,city
0,1,Lisa Brown,jonesdonna@example.net,+1-351-404-6368x15798,Basic,2022-10-30,39,Castrotown
1,2,Thomas Torres,klong@example.org,(655)543-8404x608,Basic,2022-08-05,68,Sandovalland
2,3,Kelly Gregory,robertberry@example.com,495-878-2329x8324,Gold,2026-10-25,18,Lake Sarah
3,4,Emily Grant,whitemary@example.org,001-205-666-4742x3122,Silver,2024-11-11,72,New Jesse
4,5,Heather Watts,manderson@example.com,001-252-667-7223x306,Basic,2026-08-03,26,Lake Emily
5,6,Daniel Thompson,fcamacho@example.net,273-366-0965x69998,Gold,2024-02-04,34,East Aprilfurt
6,7,Mrs. Jasmine Hubbard,leescott@example.com,001-347-426-4908x27718,Basic,2025-09-23,25,South Robert
7,8,Christopher Garza,nicoledavis@example.com,001-421-811-7518x689,Gold,2024-08-02,73,Port Georgemouth
8,9,Mary Hernandez,haroldjoseph@example.com,947-942-1230x9407,Basic,2024-07-20,26,New Nicolebury
9,10,Matthew Chavez,mmiller@example.org,284.798.3022,Basic,2022-01-07,40,Lake Belindaport


In [ ]:
# Check member records

cursor.execute("SELECT COUNT(*) FROM Members")
total_members = cursor.fetchone()[0]

cursor.execute("SELECT COUNT(DISTINCT email) FROM Members")
unique_emails = cursor.fetchone()[0]

cursor.execute("""
    SELECT COUNT(*)
    FROM Members
    WHERE member_name IS NULL
       OR email IS NULL
       OR membership_type IS NULL
       OR join_date IS NULL
       OR age IS NULL
       OR city IS NULL
""")

missing_members = cursor.fetchone()[0]

print("Total Members:", total_members)
print("Unique Emails:", unique_emails)
print("Missing Mandatory Values:", missing_members)

# Display membership distribution
membership_summary = pd.read_sql_query("""
    SELECT
        membership_type,
        COUNT(*) AS total_members
    FROM Members
    GROUP BY membership_type
    ORDER BY total_members DESC
""", conn)

display(membership_summary)

Total Members: 250
Unique Emails: 250
Missing Mandatory Values: 0


,membership_type,total_members
0,Silver,90
1,Gold,88
2,Basic,72


In [ ]:
# Generate borrowing transactions

today = date.today()

# Get existing book and member IDs
book_ids = pd.read_sql_query(
    "SELECT book_id FROM Books", conn
)["book_id"].tolist()

member_ids = pd.read_sql_query(
    "SELECT member_id FROM Members", conn
)["member_id"].tolist()

borrowings_data = []

# Generate 1500 borrowing records
for i in range(1500):

    book_id = random.choice(book_ids)
    member_id = random.choice(member_ids)

    # Select borrowing status
    status = random.choices(
        ["Returned", "Overdue", "Borrowed"],
        weights=[84, 15, 1]
    )[0]

    # Generate dates based on borrowing status
    if status == "Overdue":

        borrow_date = today - timedelta(
            days=random.randint(30, 730)
        )

        loan_days = random.choice([14, 21, 28])

        due_date = borrow_date + timedelta(days=loan_days)

        return_date = None

        overdue_days = (today - due_date).days

        fine_amount = min(overdue_days * 5, 500)

    elif status == "Returned":

        borrow_date = today - timedelta(
            days=random.randint(10, 730)
        )

        loan_days = random.choice([14, 21, 28])

        due_date = borrow_date + timedelta(days=loan_days)

        # Generate a return date after borrowing
        return_date = borrow_date + timedelta(
            days=random.randint(1, (today - borrow_date).days)
        )

        # Calculate fine for late returns
        overdue_days = max((return_date - due_date).days, 0)

        fine_amount = min(overdue_days * 5, 500)

    else:

        # Generate recent borrowing dates
        borrow_date = today - timedelta(
            days=random.randint(0, 10)
        )

        loan_days = random.choice([14, 21, 28])

        due_date = borrow_date + timedelta(days=loan_days)

        return_date = None
        fine_amount = 0

    # Add the borrowing details to the list
    borrowing = (
        book_id,
        member_id,
        borrow_date.isoformat(),
        due_date.isoformat(),
        return_date.isoformat() if return_date else None,
        fine_amount,
        status,
        random.randint(0, 2)
    )

    borrowings_data.append(borrowing)


# Insert borrowing records into SQLite
cursor.executemany("""
    INSERT INTO Borrowings (
        book_id,
        member_id,
        borrow_date,
        due_date,
        return_date,
        fine_amount,
        borrowing_status,
        renewal_count
    )
    VALUES (?, ?, ?, ?, ?, ?, ?, ?)
""", borrowings_data)

conn.commit()

print("Borrowing records generated successfully")
print("Total Borrowings:", len(borrowings_data))

Borrowing records generated successfully!
Total Borrowings: 1500


In [ ]:
# Display sample borrowing records

borrowings_df = pd.read_sql_query("""
    SELECT *
    FROM Borrowings
    LIMIT 10
""", conn)

display(borrowings_df)

,book_id,member_id,borrow_date,due_date,return_date,fine_amount,borrowing_status,renewal_count
0,166,231,2025-05-02,2025-05-16,2026-04-12,500.0,Returned,2
1,87,90,2026-05-10,2026-05-24,2026-10-17,500.0,Returned,0
2,147,189,2024-11-30,2024-12-28,2025-08-01,500.0,Returned,1
3,184,27,2025-07-21,2025-08-04,2026-04-21,500.0,Returned,0
4,8,129,2025-06-30,2025-07-14,2026-01-05,500.0,Returned,2
5,182,75,2026-01-08,2026-01-29,2026-09-23,500.0,Returned,2
6,234,129,2026-04-02,2026-04-30,2026-05-08,40.0,Returned,1
7,159,242,2025-10-08,2025-11-05,2025-11-30,125.0,Returned,2
8,284,107,2026-07-30,2026-08-13,2026-09-03,105.0,Returned,1
9,243,88,2026-07-22,2026-08-05,2026-09-18,220.0,Returned,1


Data check

In [ ]:
# Analyse borrowing status

status_summary = pd.read_sql_query("""
    SELECT
        borrowing_status,
        COUNT(*) AS total_records,
        ROUND(SUM(fine_amount), 2) AS total_fines,
        ROUND(AVG(fine_amount), 2) AS average_fine
    FROM Borrowings
    GROUP BY borrowing_status
""", conn)

display(status_summary)

,borrowing_status,total_records,total_fines,average_fine
0,Borrowed,12,0.0,0.00
1,Overdue,206,97585.0,473.71
2,Returned,1282,454620.0,354.62


In [ ]:
# Display the total number of records in each table

database_summary = pd.read_sql_query("""
    SELECT 'Books' AS table_name, COUNT(*) AS total_records
    FROM Books

    UNION ALL

    SELECT 'Members', COUNT(*)
    FROM Members

    UNION ALL

    SELECT 'Borrowings', COUNT(*)
    FROM Borrowings
""", conn)

display(database_summary)

# Calculate total database records
total_records = database_summary["total_records"].sum()

print("Total Database Records:", total_records)

,table_name,total_records
0,Books,300
1,Members,250
2,Borrowings,1500


Total Database Records: 2050


In [ ]:
# Check foreign key relationships in Borrowings

cursor.execute("PRAGMA foreign_key_list(Borrowings)")

foreign_keys = cursor.fetchall()

print("Foreign Key Relationships:")
for key in foreign_keys:
    print(key[2], ":", key[3], "->", key[4])

# Check for foreign key errors
cursor.execute("PRAGMA foreign_key_check")

errors = cursor.fetchall()

print("\nForeign Key Errors:", errors)

Foreign Key Relationships:
Members : member_id -> member_id
Books : book_id -> book_id

Foreign Key Errors: []


In [ ]:
# Check overall database integrity

cursor.execute("PRAGMA integrity_check")

integrity_result = cursor.fetchone()[0]

print("Database Integrity:", integrity_result)

Database Integrity: ok


## SQL queries for data retrieval

In [ ]:
# Retrieve borrowing details with member and book information

query1 = pd.read_sql_query("""
    SELECT
        m.member_name,
        b.title,
        br.borrow_date,
        br.due_date,
        br.borrowing_status,
        br.fine_amount
    FROM Borrowings br

    INNER JOIN Members m
        ON br.member_id = m.member_id

    INNER JOIN Books b
        ON br.book_id = b.book_id

    LIMIT 20
""", conn)

display(query1)

,member_name,title,borrow_date,due_date,borrowing_status,fine_amount
0,Wayne Porter,Admit attack,2025-05-02,2025-05-16,Returned,500.0
1,Nancy Dorsey,Affect democratic,2026-05-10,2026-05-24,Returned,500.0
2,Sara Thomas,Leg quickly real character,2024-11-30,2024-12-28,Returned,500.0
3,Jonathan Simon,Center way student special wife,2025-07-21,2025-08-04,Returned,500.0
4,Karen Sosa,Participant check several much,2025-06-30,2025-07-14,Returned,500.0
5,Helen Hernandez,Establish ago others ahead,2026-01-08,2026-01-29,Returned,500.0
6,Karen Sosa,Guess despite sell most,2026-04-02,2026-04-30,Returned,40.0
7,Erica Hale,Follow exist natural,2025-10-08,2025-11-05,Returned,125.0
8,Brianna Martinez,Small series south just cover,2026-07-30,2026-08-13,Returned,105.0
9,Billy Tate,Size Republican political,2026-07-22,2026-08-05,Returned,220.0


In [ ]:
# Find the top 10 most borrowed books

query2 = pd.read_sql_query("""
    SELECT
        b.title,
        b.author_name,
        COUNT(*) AS total_borrowings
    FROM Borrowings br

    INNER JOIN Books b
        ON br.book_id = b.book_id

    GROUP BY b.book_id, b.title, b.author_name

    ORDER BY total_borrowings DESC

    LIMIT 10
""", conn)

display(query2)

,title,author_name,total_borrowings
0,Truth hold simple quickly,Mark Cook,12
1,Society behavior develop reality,Kevin Kennedy,11
2,Foreign fast,Mitchell Jackson,11
3,Partner area program price,Adam Taylor,11
4,Establish ago others ahead,Diana Stokes,11
5,Center way student special wife,Casey Johnson,11
6,Question consider here grow agreement,Sara Mcpherson,11
7,Different chance enter central arrive,Linda Holloway,10
8,Huge couple,Ryan Gibson,10
9,Last cell anything,Sharon Barnes,10


In [ ]:
# Count members in each membership category

query3 = pd.read_sql_query("""
    SELECT
        membership_type,
        COUNT(*) AS total_members
    FROM Members

    GROUP BY membership_type

    ORDER BY total_members DESC
""", conn)

display(query3)

,membership_type,total_members
0,Silver,90
1,Gold,88
2,Basic,72


In [ ]:
# Calculate total fines for each borrowing status

query4 = pd.read_sql_query("""
    SELECT
        borrowing_status,
        COUNT(*) AS total_records,
        SUM(fine_amount) AS total_fines,
        ROUND(AVG(fine_amount), 2) AS average_fine
    FROM Borrowings

    GROUP BY borrowing_status
""", conn)

display(query4)

,borrowing_status,total_records,total_fines,average_fine
0,Borrowed,12,0.0,0.00
1,Overdue,206,97585.0,473.71
2,Returned,1282,454620.0,354.62


In [ ]:
# Find members who have overdue books

query5 = pd.read_sql_query("""
    SELECT
        m.member_name,
        m.email,
        b.title,
        br.borrow_date,
        br.due_date,
        br.fine_amount
    FROM Borrowings br

    INNER JOIN Members m
        ON br.member_id = m.member_id

    INNER JOIN Books b
        ON br.book_id = b.book_id

    WHERE br.borrowing_status = 'Overdue'

    ORDER BY br.fine_amount DESC
""", conn)

# Display the first 20 overdue records
display(query5.head(20))

,member_name,email,title,borrow_date,due_date,fine_amount
0,Jennifer Cohen,martinkaren@example.com,Church find food walk,2025-01-18,2025-02-15,500.0
1,Joel Boyer,sadams@example.com,Stage field exactly man,2025-03-15,2025-03-29,500.0
2,Jeffrey Cook,gnelson@example.org,They treatment personal interview many,2025-02-18,2025-03-18,500.0
3,Jennifer Martinez,hsimpson@example.org,Ago listen whose situation simply,2025-06-20,2025-07-18,500.0
4,Elizabeth Richards,dhansen@example.org,Middle discuss college already dream,2025-12-22,2026-01-19,500.0
5,Dawn Jones,william50@example.org,Partner area program price,2025-12-03,2025-12-24,500.0
6,Nathan Chen,james20@example.net,Popular even know television guy,2024-12-01,2024-12-22,500.0
7,Brandy Miller,ostone@example.net,Right couple environmental purpose owner,2025-06-03,2025-06-24,500.0
8,Kelly Gregory,robertberry@example.com,Oil worker although,2025-11-09,2025-11-30,500.0
9,Benjamin Neal,vjames@example.com,Large instead everything economic type,2025-08-02,2025-08-16,500.0


Data Validation

In [ ]:
# Check for missing values in all three tables

for table in ["Books", "Members", "Borrowings"]:

    data = pd.read_sql_query(
        f"SELECT * FROM {table}", conn
    )

    print("\nTable:", table)
    print("Total Records:", len(data))
    print("Missing Values:", data.isnull().sum().sum())


Table: Books
Total Records: 300
Missing Values: 0

Table: Members
Total Records: 250
Missing Values: 0

Table: Borrowings
Total Records: 1500
Missing Values: 218


In [ ]:
# Check foreign key constraints
cursor.execute("PRAGMA foreign_key_check")

foreign_key_errors = cursor.fetchall()

# Check database integrity
cursor.execute("PRAGMA integrity_check")

integrity_result = cursor.fetchone()[0]

print("Foreign Key Errors:", foreign_key_errors)
print("Database Integrity:", integrity_result)

if len(foreign_key_errors) == 0 and integrity_result == "ok":
    print("Database validation completed successfully")
else:
    print("Please check the database for errors.")

Foreign Key Errors: []
Database Integrity: ok
Database validation completed successfully


Data Saved

In [ ]:
# Save all database changes
conn.commit()

# Close the database connection
conn.close()

print("Database saved and connection closed.")

# Download the final SQLite database
files.download(db_file)

Database saved and connection closed.


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>